# Семинар 2. Исследование методов линейной регрессии

#### Критерий оценивания:
#### Пункт 14: максимум 0,5 балла
#### Пункт 15: максимум 0,5 балла

#### Итого за работу: максимум 1 балл
#### P.S. пункты 14 и 15 без пунктов 1-13 не засчитываются, ответы на вопросы из ИИ не засчитываются

1. Загрузите датасет и выведите на экран первые несколько строк

In [78]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.metrics import r2_score

# Загрузка датасета
data = pd.read_csv('auto_dataset.csv')
data.head()

,brand,model,vehicleType,gearbox,fuelType,notRepairedDamage,powerPS,kilometer,autoAgeMonths,price
0,volkswagen,golf,kleinwagen,manuell,benzin,nein,75,150000,177,1500
1,skoda,fabia,kleinwagen,manuell,diesel,nein,69,90000,93,3600
2,bmw,3er,limousine,manuell,benzin,ja,102,150000,246,650
3,peugeot,2_reihe,cabrio,manuell,benzin,nein,109,150000,140,2200
4,mazda,3_reihe,limousine,manuell,benzin,nein,105,150000,136,2000


2. Разбейте выборку на признаки и ответы. Закодируйте категориальные признаки.

In [79]:

X = data.drop('price', axis=1)
y = data['price']

print("X:", X.shape)
print("y:", y.shape)

X: (1000, 9)
y: (1000,)


3. Разбейте датасет на train val test в отношении 8:1:1

In [80]:

X_train, X_temp, y_train, y_temp = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42
)

X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp,
    test_size=0.5,
    random_state=42
)

print("Train:", X_train.shape, y_train.shape)
print("Val:  ", X_val.shape,   y_val.shape)
print("Test: ", X_test.shape,  y_test.shape)

cat_cols = ['brand', 'model', 'vehicleType', 'gearbox', 'fuelType', 'notRepairedDamage']
num_cols = ['powerPS', 'kilometer', 'autoAgeMonths']

preprocessor = ColumnTransformer(
    transformers=[
        ('cat', OneHotEncoder(handle_unknown='ignore'), cat_cols),
        ('num', StandardScaler(), num_cols)
    ]
)

# fit ТОЛЬКО на train
X_train_enc = preprocessor.fit_transform(X_train)
X_val_enc   = preprocessor.transform(X_val)
X_test_enc  = preprocessor.transform(X_test)

print("X_train_enc:", X_train_enc.shape)
print("X_val_enc:  ", X_val_enc.shape)
print("X_test_enc: ", X_test_enc.shape)

Train: (800, 9) (800,)
Val:   (100, 9) (100,)
Test:  (100, 9) (100,)
X_train_enc: (800, 194)
X_val_enc:   (100, 194)
X_test_enc:  (100, 194)


4. Исследуйте VGD с постоянным шагом n:

переберите n в логарифмической сетке от 10^-5 до 1

для каждого n:

* обучите VGD на train
* найдите и запомните R^2_train и Loss_train
* протестируйте VGD на val, запомните Loss_val

найдите наилучший n по минимальному Loss_val, запомните его Loss_train, R^2_train, Loss_val;

протестируйте VGD c лучшим n на test, запомните Loss_test, R^2_test, число итераций на test.

In [81]:
class VGD:
    def __init__(self, learning_rate=0.01, n_iters=3000, tol=1e-8):
        self.learning_rate = learning_rate
        self.n_iters = n_iters
        self.tol = tol
        self.w = None
        self.loss_history = []
        self.n_iter_done = None

    def fit(self, X, y):
        n_samples, n_features = X.shape
        self.w = np.zeros(n_features)
        y = np.asarray(y, dtype=float)
        self.loss_history = []
        self.n_iter_done = self.n_iters

        for i in range(self.n_iters):
            y_pred = np.asarray(X @ self.w).ravel()
            error = y_pred - y
            grad = (2.0 / n_samples) * np.asarray(X.T @ error).ravel()

            self.w -= self.learning_rate * grad

            loss = float(np.mean(error ** 2))
            self.loss_history.append(loss)

            if not np.isfinite(loss):
                self.loss_history.append(np.inf)
                self.n_iter_done = i + 1
                break

            if np.linalg.norm(grad) < self.tol:
                self.n_iter_done = i + 1
                break

        return self

    def predict(self, X):
        return np.asarray(X @ self.w).ravel()

    def score(self, X, y):
        y = np.asarray(y, dtype=float)
        y_pred = self.predict(X)
        ss_res = np.sum((y - y_pred) ** 2)
        ss_tot = np.sum((y - y.mean()) ** 2)
        return 1.0 - ss_res / ss_tot

In [82]:
lambdas = np.logspace(-5, 0, 11)

results_vgd = []

for n in lambdas:
    model = VGD(learning_rate=n, n_iters=3000, tol=1e-8)
    model.fit(X_train_enc, y_train)

    # ---- train ----
    y_train_pred = model.predict(X_train_enc)
    if np.all(np.isfinite(y_train_pred)):
        loss_train = float(np.mean((y_train - y_train_pred) ** 2))
        r2_train   = r2_score(y_train, y_train_pred)
    else:
        loss_train, r2_train = np.inf, np.nan

    # ---- val ----
    y_val_pred = model.predict(X_val_enc)
    if np.all(np.isfinite(y_val_pred)):
        loss_val = float(np.mean((y_val - y_val_pred) ** 2))
    else:
        loss_val = np.inf

    results_vgd.append({
        'n':          n,
        'Loss_train': loss_train,
        'R2_train':   r2_train,
        'Loss_val':   loss_val,
    })

results_vgd_df = pd.DataFrame(results_vgd)
pd.set_option('display.float_format', lambda x: f'{x:.6g}')
print(results_vgd_df.to_string(index=False))

          n  Loss_train   R2_train    Loss_val
      1e-05  9.2031e+07  -0.512116 8.98762e+07
3.16228e-05 6.60701e+07 -0.0855647 6.38467e+07
     0.0001 3.31413e+07   0.455472 3.15078e+07
0.000316228 2.08772e+07   0.656977 2.17713e+07
      0.001   1.876e+07   0.691763 2.15652e+07
 0.00316228 1.74893e+07   0.712642 2.05242e+07
       0.01 1.59975e+07   0.737154 1.90426e+07
  0.0316228 1.42449e+07   0.765949 1.74255e+07
        0.1 1.29684e+07   0.786923 1.66942e+07
   0.316228 1.23414e+07   0.797224 1.67479e+07
          1         inf       -inf         inf


/usr/lib/python3.14/site-packages/numpy/_core/_methods.py:132: RuntimeWarning: overflow encountered in reduce
  ret = umr_sum(arr, axis, dtype, out, keepdims, where=where)
/usr/lib/python3.14/site-packages/numpy/_core/_methods.py:49: RuntimeWarning: overflow encountered in reduce
  return umr_sum(a, axis, dtype, out, keepdims, initial, where)
/usr/lib/python3.14/site-packages/numpy/_core/fromnumeric.py:83: RuntimeWarning: overflow encountered in reduce
  return ufunc.reduce(obj, axis, dtype, out, **passkwargs)
/usr/lib/python3.14/site-packages/numpy/_core/_methods.py:49: RuntimeWarning: overflow encountered in reduce
  return umr_sum(a, axis, dtype, out, keepdims, initial, where)


In [83]:
best_row_vgd    = results_vgd_df.loc[results_vgd_df['Loss_val'].idxmin()]
best_n_vgd      = best_row_vgd['n']
best_loss_train = best_row_vgd['Loss_train']
best_r2_train   = best_row_vgd['R2_train']
best_loss_val   = best_row_vgd['Loss_val']

print(f"Лучший n   = {best_n_vgd:.6g}")
print(f"Loss_train = {best_loss_train:.6f}")
print(f"R2_train   = {best_r2_train:.4f}")
print(f"Loss_val   = {best_loss_val:.6f}")

Лучший n   = 0.1
Loss_train = 12968358.210505
R2_train   = 0.7869
Loss_val   = 16694164.549343


In [84]:

best_vgd = VGD(learning_rate=best_n_vgd, n_iters=3000, tol=1e-8)
best_vgd.fit(X_train_enc, y_train)

y_test_pred_vgd = best_vgd.predict(X_test_enc)
if np.all(np.isfinite(y_test_pred_vgd)):
    loss_test_vgd = float(np.mean((y_test - y_test_pred_vgd) ** 2))
    r2_test_vgd   = r2_score(y_test, y_test_pred_vgd)
else:
    loss_test_vgd, r2_test_vgd = np.inf, np.nan

n_iter_vgd = best_vgd.n_iter_done

print("=== VGD, лучший n ===")
print(f"n          = {best_n_vgd:.6g}")
print(f"Loss_train = {best_loss_train:.6f}")
print(f"R2_train   = {best_r2_train:.4f}")
print(f"Loss_val   = {best_loss_val:.6f}")
print(f"Loss_test  = {loss_test_vgd:.6f}")
print(f"R2_test    = {r2_test_vgd:.4f}")
print(f"Итераций   = {n_iter_vgd}")

=== VGD, лучший n ===
n          = 0.1
Loss_train = 12968358.210505
R2_train   = 0.7869
Loss_val   = 16694164.549343
Loss_test  = 27313178.637145
R2_test    = 0.6083
Итераций   = 3000


In [85]:
best_vgd_model   = best_vgd
best_vgd_n       = best_n_vgd
best_vgd_loss_tr = best_loss_train
best_vgd_loss_te = loss_test_vgd
best_vgd_r2_tr   = best_r2_train
best_vgd_r2_te   = r2_test_vgd
best_vgd_iters   = n_iter_vgd

5. Исследуйте VGD с переменным шагом n(lyamda) по формуле TimeDecayLR (из лекции):

переберите lyamda в логарифмической сетке от 10^-5 до 1

для каждого lyamda:

* обучите VGD на train
* найдите и запомните R^2_train и Loss_train
* протестируйте VGD на val, запомните Loss_val

найдите наилучший lyamda по минимальному Loss_val, запомните его Loss_train, R^2_train, Loss_val;

протестируйте VGD c n(best_lyamda) на test, запомните Loss_test, R^2_test, число итераций на test.

In [86]:
class VGDTimeDecay:
    def __init__(self, n0=0.1, lam=0.01, n_iters=3000, tol=1e-8):
        self.n0 = n0             
        self.lam = lam       
        self.n_iters = n_iters
        self.tol = tol
        self.w = None
        self.loss_history = []
        self.n_iter_done = None

    def fit(self, X, y):
        n_samples, n_features = X.shape
        self.w = np.zeros(n_features)
        y = np.asarray(y, dtype=float)
        self.loss_history = []
        self.n_iter_done = self.n_iters

        for t in range(1, self.n_iters + 1):
            lr_t = self.n0 / (1.0 + self.lam * t)    

            y_pred = np.asarray(X @ self.w).ravel()
            error = y_pred - y
            grad = (2.0 / n_samples) * np.asarray(X.T @ error).ravel()

            self.w -= lr_t * grad

            loss = float(np.mean(error ** 2))
            self.loss_history.append(loss)

            if not np.isfinite(loss):
                self.loss_history.append(np.inf)
                self.n_iter_done = t
                break

            if np.linalg.norm(grad) < self.tol:
                self.n_iter_done = t
                break

        return self

    def predict(self, X):
        return np.asarray(X @ self.w).ravel()

    def score(self, X, y):
        y = np.asarray(y, dtype=float)
        y_pred = self.predict(X)
        ss_res = np.sum((y - y_pred) ** 2)
        ss_tot = np.sum((y - y.mean()) ** 2)
        return 1.0 - ss_res / ss_tot

In [87]:
lambdas = np.logspace(-5, 0, 11)

n0 = 0.1

results_td = []

for lam in lambdas:
    model = VGDTimeDecay(n0=n0, lam=lam, n_iters=3000, tol=1e-8)
    model.fit(X_train_enc, y_train)

    # ---- train ----
    y_train_pred = model.predict(X_train_enc)
    if np.all(np.isfinite(y_train_pred)):
        loss_train = float(np.mean((y_train - y_train_pred) ** 2))
        r2_train   = r2_score(y_train, y_train_pred)
    else:
        loss_train, r2_train = np.inf, np.nan

    # ---- val ----
    y_val_pred = model.predict(X_val_enc)
    if np.all(np.isfinite(y_val_pred)):
        loss_val = float(np.mean((y_val - y_val_pred) ** 2))
    else:
        loss_val = np.inf

    results_td.append({
        'lyamda':     lam,
        'Loss_train': loss_train,
        'R2_train':   r2_train,
        'Loss_val':   loss_val,
    })

results_td_df = pd.DataFrame(results_td)
pd.set_option('display.float_format', lambda x: f'{x:.6g}')
print(results_td_df.to_string(index=False))

     lyamda  Loss_train  R2_train    Loss_val
      1e-05 1.29803e+07  0.786727 1.66965e+07
3.16228e-05 1.30055e+07  0.786313 1.67019e+07
     0.0001 1.30807e+07  0.785077 1.67213e+07
0.000316228  1.3285e+07  0.781721 1.67959e+07
      0.001 1.37505e+07  0.774072 1.70572e+07
 0.00316228 1.45962e+07  0.760177 1.77225e+07
       0.01 1.57921e+07  0.740529 1.88426e+07
  0.0316228 1.70135e+07  0.720459 2.00348e+07
        0.1  1.8028e+07   0.70379 2.10811e+07
   0.316228 1.92507e+07  0.683702 2.17074e+07
          1 2.16939e+07  0.643559 2.20399e+07


In [88]:
best_row_td = results_td_df.loc[results_td_df['Loss_val'].idxmin()]
best_lyamda_td = best_row_td['lyamda']
best_loss_train_td = best_row_td['Loss_train']
best_r2_train_td   = best_row_td['R2_train']
best_loss_val_td   = best_row_td['Loss_val']

print(f"Лучший lyamda = {best_lyamda_td:.6g}")
print(f"Loss_train    = {best_loss_train_td:.6f}")
print(f"R2_train      = {best_r2_train_td:.4f}")
print(f"Loss_val      = {best_loss_val_td:.6f}")

Лучший lyamda = 1e-05
Loss_train    = 12980289.219378
R2_train      = 0.7867
Loss_val      = 16696514.639873


In [89]:
best_td = VGDTimeDecay(n0=n0, lam=best_lyamda_td, n_iters=3000, tol=1e-8)
best_td.fit(X_train_enc, y_train)

y_test_pred_td = best_td.predict(X_test_enc)
if np.all(np.isfinite(y_test_pred_td)):
    loss_test_td = float(np.mean((y_test - y_test_pred_td) ** 2))
    r2_test_td   = r2_score(y_test, y_test_pred_td)
else:
    loss_test_td, r2_test_td = np.inf, np.nan

n_iter_td = best_td.n_iter_done

print("=== VGD + TimeDecayLR ===")
print(f"n0            = {n0}")
print(f"best lyamda   = {best_lyamda_td:.6g}")
print(f"Loss_train    = {best_loss_train_td:.6f}")
print(f"R2_train      = {best_r2_train_td:.4f}")
print(f"Loss_val      = {best_loss_val_td:.6f}")
print(f"Loss_test     = {loss_test_td:.6f}")
print(f"R2_test       = {r2_test_td:.4f}")
print(f"Итераций      = {n_iter_td}")

=== VGD + TimeDecayLR ===
n0            = 0.1
best lyamda   = 1e-05
Loss_train    = 12980289.219378
R2_train      = 0.7867
Loss_val      = 16696514.639873
Loss_test     = 27297845.899933
R2_test       = 0.6086
Итераций      = 3000


In [90]:
best_td_model   = best_td
best_td_lyamda  = best_lyamda_td
best_td_loss_tr = best_loss_train_td
best_td_loss_te = loss_test_td
best_td_r2_tr   = best_r2_train_td
best_td_r2_te   = r2_test_td
best_td_iters   = n_iter_td

6. Исследуйте SGD с постоянным шагом n:

переберите n в логарифмической сетке от 10^-5 до 1

для каждого n:

* обучите SGD на train
* найдите и запомните R^2_train и Loss_train
* протестируйте SGD на val, запомните Loss_val

найдите наилучший n по минимальному Loss_val, запомните его Loss_train, R^2_train, Loss_val;

протестируйте SGD c лучшим n на test, запомните Loss_test, R^2_test, число итераций на test.

In [91]:
class SGD:
    def __init__(self, learning_rate=0.01, n_epochs=50, random_state=42):
        self.learning_rate = learning_rate
        self.n_epochs = n_epochs
        self.random_state = random_state
        self.w = None
        self.loss_history = []
        self.n_epochs_done = None

    def fit(self, X, y):
        rng = np.random.default_rng(self.random_state)
        n_samples, n_features = X.shape
        self.w = np.zeros(n_features)
        y = np.asarray(y, dtype=float)
        self.loss_history = []
        self.n_epochs_done = self.n_epochs

        is_sparse = hasattr(X, "toarray")

        for epoch in range(self.n_epochs):
            indices = rng.permutation(n_samples)

            for i in indices:
                x_i = X[i]
                if is_sparse:
                    x_i = np.asarray(x_i.toarray()).ravel()
                else:
                    x_i = np.asarray(x_i, dtype=float).ravel()

                err = float(x_i @ self.w - y[i])
                grad = 2.0 * err * x_i
                self.w -= self.learning_rate * grad

                if not np.all(np.isfinite(self.w)):
                    self.loss_history.append(np.inf)
                    self.n_epochs_done = epoch + 1
                    return self

            y_pred = np.asarray(X @ self.w).ravel()
            loss = float(np.mean((y - y_pred) ** 2))
            self.loss_history.append(loss)

            if not np.isfinite(loss):
                self.loss_history.append(np.inf)
                self.n_epochs_done = epoch + 1
                break

        return self

    def predict(self, X):
        return np.asarray(X @ self.w).ravel()

    def score(self, X, y):
        y = np.asarray(y, dtype=float)
        y_pred = self.predict(X)
        ss_res = np.sum((y - y_pred) ** 2)
        ss_tot = np.sum((y - y.mean()) ** 2)
        return 1.0 - ss_res / ss_tot

In [92]:
lambdas = np.logspace(-5, 0, 11)   

results_sgd = []

for n in lambdas:
    model = SGD(learning_rate=n, n_epochs=50, random_state=42)
    model.fit(X_train_enc, y_train)

    # ---- train ----
    y_train_pred = model.predict(X_train_enc)
    if np.all(np.isfinite(y_train_pred)):
        loss_train = float(np.mean((y_train - y_train_pred) ** 2))
        r2_train   = r2_score(y_train, y_train_pred)
    else:
        loss_train, r2_train = np.inf, np.nan

    # ---- val ----
    y_val_pred = model.predict(X_val_enc)
    if np.all(np.isfinite(y_val_pred)):
        loss_val = float(np.mean((y_val - y_val_pred) ** 2))
    else:
        loss_val = np.inf

    results_sgd.append({
        'n':          n,
        'Loss_train': loss_train,
        'R2_train':   r2_train,
        'Loss_val':   loss_val,
        'epochs':     len(model.loss_history),
    })

results_sgd_df = pd.DataFrame(results_sgd)
pd.set_option('display.float_format', lambda x: f'{x:.6g}')
print(results_sgd_df.to_string(index=False))

          n  Loss_train    R2_train    Loss_val  epochs
      1e-05 2.77765e+07    0.543618 2.66094e+07      50
3.16228e-05   2.016e+07    0.668761 2.17229e+07      50
     0.0001 1.84051e+07    0.697595 2.13935e+07      50
0.000316228 1.71637e+07    0.717993 2.02455e+07      50
      0.001 1.55439e+07    0.744607 1.87256e+07      50
 0.00316228 1.39555e+07    0.770705 1.67709e+07      50
       0.01 1.37377e+07    0.774283 1.53664e+07      50
  0.0316228 1.97394e+07    0.675672 1.91015e+07      50
        0.1 3.86811e+73 -6.3555e+65 3.89039e+73      50
   0.316228         inf        -inf         inf       3
          1         inf         NaN         inf       1


/tmp/ipykernel_4898/2812723768.py:40: RuntimeWarning: overflow encountered in square
  loss = float(np.mean((y - y_pred) ** 2))
/usr/lib/python3.14/site-packages/sklearn/metrics/_regression.py:1304: RuntimeWarning: overflow encountered in square
  numerator = xp.sum(weight * (y_true - y_pred) ** 2, axis=0)
/tmp/ipykernel_4898/2812723768.py:30: RuntimeWarning: overflow encountered in matmul
  err = float(x_i @ self.w - y[i])
/tmp/ipykernel_4898/2812723768.py:31: RuntimeWarning: invalid value encountered in multiply
  grad = 2.0 * err * x_i


In [93]:
best_row_sgd       = results_sgd_df.loc[results_sgd_df['Loss_val'].idxmin()]
best_n_sgd         = best_row_sgd['n']
best_loss_train_sgd = best_row_sgd['Loss_train']
best_r2_train_sgd   = best_row_sgd['R2_train']
best_loss_val_sgd   = best_row_sgd['Loss_val']

print(f"Лучший n   = {best_n_sgd:.6g}")
print(f"Loss_train = {best_loss_train_sgd:.6f}")
print(f"R2_train   = {best_r2_train_sgd:.4f}")
print(f"Loss_val   = {best_loss_val_sgd:.6f}")

Лучший n   = 0.01
Loss_train = 13737684.423549
R2_train   = 0.7743
Loss_val   = 15366438.475242


In [94]:
best_sgd = SGD(learning_rate=best_n_sgd, n_epochs=50, random_state=42)
best_sgd.fit(X_train_enc, y_train)

y_test_pred_sgd = best_sgd.predict(X_test_enc)
if np.all(np.isfinite(y_test_pred_sgd)):
    loss_test_sgd = float(np.mean((y_test - y_test_pred_sgd) ** 2))
    r2_test_sgd   = r2_score(y_test, y_test_pred_sgd)
else:
    loss_test_sgd, r2_test_sgd = np.inf, np.nan

n_iter_sgd = best_sgd.n_epochs_done

print("=== SGD, лучший n ===")
print(f"n          = {best_n_sgd:.6g}")
print(f"Loss_train = {best_loss_train_sgd:.6f}")
print(f"R2_train   = {best_r2_train_sgd:.4f}")
print(f"Loss_val   = {best_loss_val_sgd:.6f}")
print(f"Loss_test  = {loss_test_sgd:.6f}")
print(f"R2_test    = {r2_test_sgd:.4f}")
print(f"Эпох       = {n_iter_sgd}")

=== SGD, лучший n ===
n          = 0.01
Loss_train = 13737684.423549
R2_train   = 0.7743
Loss_val   = 15366438.475242
Loss_test  = 30006734.454338
R2_test    = 0.5697
Эпох       = 50


In [95]:
best_sgd_model   = best_sgd
best_sgd_n       = best_n_sgd
best_sgd_loss_tr = best_loss_train_sgd
best_sgd_loss_te = loss_test_sgd
best_sgd_r2_tr   = best_r2_train_sgd
best_sgd_r2_te   = r2_test_sgd
best_sgd_iters   = n_iter_sgd

7. Исследуйте SGD с переменным шагом n(lyamda) по формуле TimeDecayLR (из лекции):

переберите lyamda в логарифмической сетке от 10^-5 до 1

для каждого lyamda:

* обучите SGD на train
* найдите и запомните R^2_train и Loss_train
* протестируйте SGD на val, запомните Loss_val

найдите наилучший lyamda по минимальному Loss_val, запомните его Loss_train, R^2_train, Loss_val;

протестируйте SGD c n(best_lyamda) на test, запомните Loss_test, R^2_test, число итераций на test.

In [96]:
class SGDTimeDecay:
    def __init__(self, n0=0.1, lam=0.01, n_epochs=50, random_state=42):
        self.n0 = n0
        self.lam = lam
        self.n_epochs = n_epochs
        self.random_state = random_state
        self.w = None
        self.loss_history = []
        self.n_epochs_done = None

    def fit(self, X, y):
        rng = np.random.default_rng(self.random_state)
        n_samples, n_features = X.shape
        self.w = np.zeros(n_features)
        y = np.asarray(y, dtype=float)
        self.loss_history = []
        self.n_epochs_done = self.n_epochs

        is_sparse = hasattr(X, "toarray")
        t = 0   # счётчик обновлений

        for epoch in range(self.n_epochs):
            indices = rng.permutation(n_samples)

            for i in indices:
                t += 1
                lr_t = self.n0 / (1.0 + self.lam * t)     # TimeDecay

                x_i = X[i]
                if is_sparse:
                    x_i = np.asarray(x_i.toarray()).ravel()
                else:
                    x_i = np.asarray(x_i, dtype=float).ravel()

                err = float(x_i @ self.w - y[i])
                grad = 2.0 * err * x_i
                self.w -= lr_t * grad

                if not np.all(np.isfinite(self.w)):
                    self.loss_history.append(np.inf)
                    self.n_epochs_done = epoch + 1
                    return self

            y_pred = np.asarray(X @ self.w).ravel()
            loss = float(np.mean((y - y_pred) ** 2))
            self.loss_history.append(loss)

            if not np.isfinite(loss):
                self.loss_history.append(np.inf)
                self.n_epochs_done = epoch + 1
                break

        return self

    def predict(self, X):
        return np.asarray(X @ self.w).ravel()

    def score(self, X, y):
        y = np.asarray(y, dtype=float)
        y_pred = self.predict(X)
        ss_res = np.sum((y - y_pred) ** 2)
        ss_tot = np.sum((y - y.mean()) ** 2)
        return 1.0 - ss_res / ss_tot

In [97]:
lambdas = np.logspace(-5, 0, 11)


n0 = 0.1


results_sgd_td = []

for lam in lambdas:
    model = SGDTimeDecay(n0=n0, lam=lam, n_epochs=50, random_state=42)
    model.fit(X_train_enc, y_train)

    # ---- train ----
    y_train_pred = model.predict(X_train_enc)
    if np.all(np.isfinite(y_train_pred)):
        loss_train = float(np.mean((y_train - y_train_pred) ** 2))
        r2_train   = r2_score(y_train, y_train_pred)
    else:
        loss_train, r2_train = np.inf, np.nan

    # ---- val ----
    y_val_pred = model.predict(X_val_enc)
    if np.all(np.isfinite(y_val_pred)):
        loss_val = float(np.mean((y_val - y_val_pred) ** 2))
    else:
        loss_val = np.inf

    results_sgd_td.append({
        'lyamda':     lam,
        'Loss_train': loss_train,
        'R2_train':   r2_train,
        'Loss_val':   loss_val,
        'epochs':     len(model.loss_history),
    })

results_sgd_td_df = pd.DataFrame(results_sgd_td)
pd.set_option('display.float_format', lambda x: f'{x:.6g}')
print(results_sgd_td_df.to_string(index=False))

     lyamda  Loss_train     R2_train    Loss_val  epochs
      1e-05 1.88526e+18 -3.09757e+10 1.64376e+18      50
3.16228e-05 1.20631e+08    -0.982033 1.50764e+08      50
     0.0001 1.64566e+07      0.72961 1.68739e+07      50
0.000316228 1.32493e+07     0.782307 1.55579e+07      50
      0.001  1.3087e+07     0.784975 1.63911e+07      50
 0.00316228 1.37426e+07     0.774202 1.75068e+07      50
       0.01 1.49517e+07     0.754337  1.8065e+07      50
  0.0316228 1.66441e+07     0.726528 1.94688e+07      50
        0.1 1.80363e+07     0.703655 2.09794e+07      50
   0.316228 1.94172e+07     0.680966 2.15937e+07      50
          1 2.14571e+07      0.64745  2.2332e+07      50


In [98]:
best_row_sgd_td     = results_sgd_td_df.loc[results_sgd_td_df['Loss_val'].idxmin()]
best_lyamda_sgd_td  = best_row_sgd_td['lyamda']
best_loss_train_sgd_td = best_row_sgd_td['Loss_train']
best_r2_train_sgd_td   = best_row_sgd_td['R2_train']
best_loss_val_sgd_td   = best_row_sgd_td['Loss_val']

print(f"Лучший lyamda = {best_lyamda_sgd_td:.6g}")
print(f"Loss_train    = {best_loss_train_sgd_td:.6f}")
print(f"R2_train      = {best_r2_train_sgd_td:.4f}")
print(f"Loss_val      = {best_loss_val_sgd_td:.6f}")

Лучший lyamda = 0.000316228
Loss_train    = 13249326.356845
R2_train      = 0.7823
Loss_val      = 15557875.435675


In [99]:
best_sgd_td = SGDTimeDecay(n0=n0, lam=best_lyamda_sgd_td,
                           n_epochs=50, random_state=42)
best_sgd_td.fit(X_train_enc, y_train)

y_test_pred_sgd_td = best_sgd_td.predict(X_test_enc)
if np.all(np.isfinite(y_test_pred_sgd_td)):
    loss_test_sgd_td = float(np.mean((y_test - y_test_pred_sgd_td) ** 2))
    r2_test_sgd_td   = r2_score(y_test, y_test_pred_sgd_td)
else:
    loss_test_sgd_td, r2_test_sgd_td = np.inf, np.nan

n_iter_sgd_td = best_sgd_td.n_epochs_done

print("=== SGD + TimeDecayLR ===")
print(f"n0            = {n0}")
print(f"best lyamda   = {best_lyamda_sgd_td:.6g}")
print(f"Loss_train    = {best_loss_train_sgd_td:.6f}")
print(f"R2_train      = {best_r2_train_sgd_td:.4f}")
print(f"Loss_val      = {best_loss_val_sgd_td:.6f}")
print(f"Loss_test     = {loss_test_sgd_td:.6f}")
print(f"R2_test       = {r2_test_sgd_td:.4f}")
print(f"Эпох          = {n_iter_sgd_td}")

=== SGD + TimeDecayLR ===
n0            = 0.1
best lyamda   = 0.000316228
Loss_train    = 13249326.356845
R2_train      = 0.7823
Loss_val      = 15557875.435675
Loss_test     = 28165512.576809
R2_test       = 0.5961
Эпох          = 50


In [100]:
best_sgd_td_model   = best_sgd_td
best_sgd_td_lyamda  = best_lyamda_sgd_td
best_sgd_td_loss_tr = best_loss_train_sgd_td
best_sgd_td_loss_te = loss_test_sgd_td
best_sgd_td_r2_tr   = best_r2_train_sgd_td
best_sgd_td_r2_te   = r2_test_sgd_td
best_sgd_td_iters   = n_iter_sgd_td

8. Исследуйте SAG с постоянным шагом n:

переберите n в логарифмической сетке от 10^-5 до 1

для каждого n:

* обучите SAG на train
* найдите и запомните R^2_train и Loss_train
* протестируйте SAG на val, запомните Loss_val

найдите наилучший n по минимальному Loss_val, запомните его Loss_train, R^2_train, Loss_val;

протестируйте SAG c лучшим n на test, запомните Loss_test, R^2_test, число итераций на test.

In [101]:
class SAG:
    def __init__(self, learning_rate=0.01, n_epochs=50, random_state=42):
        self.learning_rate = learning_rate
        self.n_epochs = n_epochs
        self.random_state = random_state
        self.w = None
        self.loss_history = []
        self.n_epochs_done = None

    def fit(self, X, y):
        rng = np.random.default_rng(self.random_state)
        n_samples, n_features = X.shape
        y = np.asarray(y, dtype=float)
        self.w = np.zeros(n_features)
        self.loss_history = []
        self.n_epochs_done = self.n_epochs

        is_sparse = hasattr(X, "toarray")

        grad_memory = np.zeros((n_samples, n_features))
        grad_sum = np.zeros(n_features)

        for epoch in range(self.n_epochs):
            indices = rng.permutation(n_samples)

            for i in indices:
                x_i = X[i]
                if is_sparse:
                    x_i = np.asarray(x_i.toarray()).ravel()
                else:
                    x_i = np.asarray(x_i, dtype=float).ravel()

                err = float(x_i @ self.w - y[i])
                new_grad = 2.0 * err * x_i

                grad_sum += new_grad - grad_memory[i]
                grad_memory[i] = new_grad

                avg_grad = grad_sum / n_samples
                self.w -= self.learning_rate * avg_grad

                if not np.all(np.isfinite(self.w)):
                    self.loss_history.append(np.inf)
                    self.n_epochs_done = epoch + 1
                    return self

            y_pred = np.asarray(X @ self.w).ravel()
            loss = float(np.mean((y - y_pred) ** 2))
            self.loss_history.append(loss)

            if not np.isfinite(loss):
                self.loss_history.append(np.inf)
                self.n_epochs_done = epoch + 1
                break

        return self

    def predict(self, X):
        return np.asarray(X @ self.w).ravel()

    def score(self, X, y):
        y = np.asarray(y, dtype=float)
        y_pred = self.predict(X)
        ss_res = np.sum((y - y_pred) ** 2)
        ss_tot = np.sum((y - y.mean()) ** 2)
        return 1.0 - ss_res / ss_tot

In [102]:
lambdas = np.logspace(-5, 0, 11)

results_sag = []

for n in lambdas:
    model = SAG(learning_rate=n, n_epochs=50, random_state=42)
    model.fit(X_train_enc, y_train)

    # ---- train ----
    y_train_pred = model.predict(X_train_enc)
    if np.all(np.isfinite(y_train_pred)):
        loss_train = float(np.mean((y_train - y_train_pred) ** 2))
        r2_train   = r2_score(y_train, y_train_pred)
    else:
        loss_train, r2_train = np.inf, np.nan

    # ---- val ----
    y_val_pred = model.predict(X_val_enc)
    if np.all(np.isfinite(y_val_pred)):
        loss_val = float(np.mean((y_val - y_val_pred) ** 2))
    else:
        loss_val = np.inf

    results_sag.append({
        'n':          n,
        'Loss_train': loss_train,
        'R2_train':   r2_train,
        'Loss_val':   loss_val,
        'epochs':     len(model.loss_history),
    })

results_sag_df = pd.DataFrame(results_sag)
pd.set_option('display.float_format', lambda x: f'{x:.6g}')
print(results_sag_df.to_string(index=False))

KeyboardInterrupt: 

In [ ]:
best_row_sag        = results_sag_df.loc[results_sag_df['Loss_val'].idxmin()]
best_n_sag          = best_row_sag['n']
best_loss_train_sag = best_row_sag['Loss_train']
best_r2_train_sag   = best_row_sag['R2_train']
best_loss_val_sag   = best_row_sag['Loss_val']

print(f"Лучший n   = {best_n_sag:.6g}")
print(f"Loss_train = {best_loss_train_sag:.6f}")
print(f"R2_train   = {best_r2_train_sag:.4f}")
print(f"Loss_val   = {best_loss_val_sag:.6f}")

Лучший n   = 0.001
Loss_train = 15557948.581056
R2_train   = 0.7444
Loss_val   = 18622480.750481


In [ ]:
best_sag = SAG(learning_rate=best_n_sag, n_epochs=50, random_state=42)
best_sag.fit(X_train_enc, y_train)

y_test_pred_sag = best_sag.predict(X_test_enc)
if np.all(np.isfinite(y_test_pred_sag)):
    loss_test_sag = float(np.mean((y_test - y_test_pred_sag) ** 2))
    r2_test_sag   = r2_score(y_test, y_test_pred_sag)
else:
    loss_test_sag, r2_test_sag = np.inf, np.nan

n_iter_sag = best_sag.n_epochs_done

print("=== SAG, лучший n ===")
print(f"n          = {best_n_sag:.6g}")
print(f"Loss_train = {best_loss_train_sag:.6f}")
print(f"R2_train   = {best_r2_train_sag:.4f}")
print(f"Loss_val   = {best_loss_val_sag:.6f}")
print(f"Loss_test  = {loss_test_sag:.6f}")
print(f"R2_test    = {r2_test_sag:.4f}")
print(f"Эпох       = {n_iter_sag}")

=== SAG, лучший n ===
n          = 0.001
Loss_train = 15557948.581056
R2_train   = 0.7444
Loss_val   = 18622480.750481
Loss_test  = 25136700.534158
R2_test    = 0.6396
Эпох       = 50


In [ ]:
best_sag_model   = best_sag
best_sag_n       = best_n_sag
best_sag_loss_tr = best_loss_train_sag
best_sag_loss_te = loss_test_sag
best_sag_r2_tr   = best_r2_train_sag
best_sag_r2_te   = r2_test_sag
best_sag_iters   = n_iter_sag

9. Исследуйте SAG с переменным шагом n(lyamda) по формуле TimeDecayLR (из лекции):

переберите lyamda в логарифмической сетке от 10^-5 до 1

для каждого lyamda:

* обучите SAG на train
* найдите и запомните R^2_train и Loss_train
* протестируйте SAG на val, запомните Loss_val

найдите наилучший lyamda по минимальному Loss_val, запомните его Loss_train, R^2_train, Loss_val;

протестируйте SAG c n(best_lyamda) на test, запомните Loss_test, R^2_test, число итераций на test.

In [ ]:
class SAGTimeDecay:
    def __init__(self, n0=0.1, lam=0.01, n_epochs=50, random_state=42):
        self.n0 = n0
        self.lam = lam
        self.n_epochs = n_epochs
        self.random_state = random_state
        self.w = None
        self.loss_history = []
        self.n_epochs_done = None

    def fit(self, X, y):
        rng = np.random.default_rng(self.random_state)
        n_samples, n_features = X.shape
        y = np.asarray(y, dtype=float)
        self.w = np.zeros(n_features)
        self.loss_history = []
        self.n_epochs_done = self.n_epochs

        is_sparse = hasattr(X, "toarray")

        grad_memory = np.zeros((n_samples, n_features))
        grad_sum = np.zeros(n_features)
        t = 0   

        for epoch in range(self.n_epochs):
            indices = rng.permutation(n_samples)

            for i in indices:
                t += 1
                lr_t = self.n0 / (1.0 + self.lam * t) 

                x_i = X[i]
                if is_sparse:
                    x_i = np.asarray(x_i.toarray()).ravel()
                else:
                    x_i = np.asarray(x_i, dtype=float).ravel()

                err = float(x_i @ self.w - y[i])
                new_grad = 2.0 * err * x_i

                grad_sum += new_grad - grad_memory[i]
                grad_memory[i] = new_grad

                avg_grad = grad_sum / n_samples
                self.w -= lr_t * avg_grad

                if not np.all(np.isfinite(self.w)):
                    self.loss_history.append(np.inf)
                    self.n_epochs_done = epoch + 1
                    return self

            y_pred = np.asarray(X @ self.w).ravel()
            loss = float(np.mean((y - y_pred) ** 2))
            self.loss_history.append(loss)

            if not np.isfinite(loss):
                self.loss_history.append(np.inf)
                self.n_epochs_done = epoch + 1
                break

        return self

    def predict(self, X):
        return np.asarray(X @ self.w).ravel()

    def score(self, X, y):
        y = np.asarray(y, dtype=float)
        y_pred = self.predict(X)
        ss_res = np.sum((y - y_pred) ** 2)
        ss_tot = np.sum((y - y.mean()) ** 2)
        return 1.0 - ss_res / ss_tot

In [ ]:
lambdas = np.logspace(-5, 0, 11)

n0 = 0.1


results_sag_td = []

for lam in lambdas:
    model = SAGTimeDecay(n0=n0, lam=lam, n_epochs=50, random_state=42)
    model.fit(X_train_enc, y_train)

    # ---- train ----
    y_train_pred = model.predict(X_train_enc)
    if np.all(np.isfinite(y_train_pred)):
        loss_train = float(np.mean((y_train - y_train_pred) ** 2))
        r2_train   = r2_score(y_train, y_train_pred)
    else:
        loss_train, r2_train = np.inf, np.nan

    # ---- val ----
    y_val_pred = model.predict(X_val_enc)
    if np.all(np.isfinite(y_val_pred)):
        loss_val = float(np.mean((y_val - y_val_pred) ** 2))
    else:
        loss_val = np.inf

    results_sag_td.append({
        'lyamda':     lam,
        'Loss_train': loss_train,
        'R2_train':   r2_train,
        'Loss_val':   loss_val,
        'epochs':     len(model.loss_history),
    })

results_sag_td_df = pd.DataFrame(results_sag_td)
pd.set_option('display.float_format', lambda x: f'{x:.6g}')
print(results_sag_td_df.to_string(index=False))

     lyamda  Loss_train     R2_train    Loss_val  epochs
      1e-05 1.38434e+32 -2.27453e+24 1.31874e+32      50
3.16228e-05 5.74644e+27 -9.44168e+19 4.88362e+27      50
     0.0001 6.37033e+22 -1.04668e+15 5.04748e+22      50
0.000316228 2.15012e+16 -3.53275e+08 1.85368e+16      50
      0.001 1.09875e+12     -18051.9 1.09921e+12      50
 0.00316228 1.42553e+07     0.765778 1.77579e+07      50
       0.01 1.54343e+07     0.746406 1.89375e+07      50
  0.0316228 1.70198e+07     0.720356 2.00403e+07      50
        0.1 1.82362e+07      0.70037 2.12528e+07      50
   0.316228 1.97741e+07     0.675102 2.19558e+07      50
          1  2.4238e+07     0.601757 2.36384e+07      50


In [ ]:
best_row_sag_td        = results_sag_td_df.loc[results_sag_td_df['Loss_val'].idxmin()]
best_lyamda_sag_td     = best_row_sag_td['lyamda']
best_loss_train_sag_td = best_row_sag_td['Loss_train']
best_r2_train_sag_td   = best_row_sag_td['R2_train']
best_loss_val_sag_td   = best_row_sag_td['Loss_val']

print(f"Лучший lyamda = {best_lyamda_sag_td:.6g}")
print(f"Loss_train    = {best_loss_train_sag_td:.6f}")
print(f"R2_train      = {best_r2_train_sag_td:.4f}")
print(f"Loss_val      = {best_loss_val_sag_td:.6f}")

Лучший lyamda = 0.00316228
Loss_train    = 14255333.187454
R2_train      = 0.7658
Loss_val      = 17757881.443655


In [ ]:
best_sag_td = SAGTimeDecay(n0=n0, lam=best_lyamda_sag_td,
                           n_epochs=50, random_state=42)
best_sag_td.fit(X_train_enc, y_train)

y_test_pred_sag_td = best_sag_td.predict(X_test_enc)
if np.all(np.isfinite(y_test_pred_sag_td)):
    loss_test_sag_td = float(np.mean((y_test - y_test_pred_sag_td) ** 2))
    r2_test_sag_td   = r2_score(y_test, y_test_pred_sag_td)
else:
    loss_test_sag_td, r2_test_sag_td = np.inf, np.nan

n_iter_sag_td = best_sag_td.n_epochs_done

print("=== SAG + TimeDecayLR ===")
print(f"n0            = {n0}")
print(f"best lyamda   = {best_lyamda_sag_td:.6g}")
print(f"Loss_train    = {best_loss_train_sag_td:.6f}")
print(f"R2_train      = {best_r2_train_sag_td:.4f}")
print(f"Loss_val      = {best_loss_val_sag_td:.6f}")
print(f"Loss_test     = {loss_test_sag_td:.6f}")
print(f"R2_test       = {r2_test_sag_td:.4f}")
print(f"Эпох          = {n_iter_sag_td}")

=== SAG + TimeDecayLR ===
n0            = 0.1
best lyamda   = 0.00316228
Loss_train    = 14255333.187454
R2_train      = 0.7658
Loss_val      = 17757881.443655
Loss_test     = 26935440.230901
R2_test       = 0.6138
Эпох          = 50


In [ ]:
best_sag_td_model   = best_sag_td
best_sag_td_lyamda  = best_lyamda_sag_td
best_sag_td_loss_tr = best_loss_train_sag_td
best_sag_td_loss_te = loss_test_sag_td
best_sag_td_r2_tr   = best_r2_train_sag_td
best_sag_td_r2_te   = r2_test_sag_td
best_sag_td_iters   = n_iter_sag_td

10. Исследуйте Momentum с постоянным шагом n:

переберите n в логарифмической сетке от 10^-5 до 1

для каждого n:

* обучите Momentum на train
* найдите и запомните R^2_train и Loss_train
* протестируйте Momentum на val, запомните Loss_val

найдите наилучший n по минимальному Loss_val, запомните его Loss_train, R^2_train, Loss_val;

протестируйте Momentum c лучшим n на test, запомните Loss_test, R^2_test, число итераций на test.

In [ ]:
class Momentum:
    def __init__(self, learning_rate=0.01, beta=0.9, n_iters=3000, tol=1e-8):
        self.learning_rate = learning_rate
        self.beta = beta
        self.n_iters = n_iters
        self.tol = tol
        self.w = None
        self.loss_history = []
        self.n_iter_done = None

    def fit(self, X, y):
        n_samples, n_features = X.shape
        y = np.asarray(y, dtype=float)
        self.w = np.zeros(n_features)
        v = np.zeros(n_features)
        self.loss_history = []
        self.n_iter_done = self.n_iters

        for i in range(self.n_iters):
            y_pred = np.asarray(X @ self.w).ravel()
            error = y_pred - y
            grad = (2.0 / n_samples) * np.asarray(X.T @ error).ravel()

            v = self.beta * v - self.learning_rate * grad
            self.w += v

            loss = float(np.mean(error ** 2))
            self.loss_history.append(loss)

            if not np.isfinite(loss):
                self.loss_history.append(np.inf)
                self.n_iter_done = i + 1
                break

            if np.linalg.norm(grad) < self.tol:
                self.n_iter_done = i + 1
                break

        return self

    def predict(self, X):
        return np.asarray(X @ self.w).ravel()

    def score(self, X, y):
        y = np.asarray(y, dtype=float)
        y_pred = self.predict(X)
        ss_res = np.sum((y - y_pred) ** 2)
        ss_tot = np.sum((y - y.mean()) ** 2)
        return 1.0 - ss_res / ss_tot

In [ ]:
lambdas = np.logspace(-5, 0, 11)

results_mom = []

for n in lambdas:
    model = Momentum(learning_rate=n, beta=0.9, n_iters=3000, tol=1e-8)
    model.fit(X_train_enc, y_train)

    # ---- train ----
    y_train_pred = model.predict(X_train_enc)
    if np.all(np.isfinite(y_train_pred)):
        loss_train = float(np.mean((y_train - y_train_pred) ** 2))
        r2_train   = r2_score(y_train, y_train_pred)
    else:
        loss_train, r2_train = np.inf, np.nan

    # ---- val ----
    y_val_pred = model.predict(X_val_enc)
    if np.all(np.isfinite(y_val_pred)):
        loss_val = float(np.mean((y_val - y_val_pred) ** 2))
    else:
        loss_val = np.inf

    results_mom.append({
        'n':          n,
        'Loss_train': loss_train,
        'R2_train':   r2_train,
        'Loss_val':   loss_val,
        'iters':      len(model.loss_history),
    })

results_mom_df = pd.DataFrame(results_mom)
pd.set_option('display.float_format', lambda x: f'{x:.6g}')
print(results_mom_df.to_string(index=False))

          n  Loss_train  R2_train    Loss_val  iters
      1e-05 3.31425e+07  0.455452 3.15063e+07   3000
3.16228e-05 2.08737e+07  0.657034 2.17715e+07   3000
     0.0001  1.8761e+07  0.691747 2.15705e+07   3000
0.000316228 1.74913e+07  0.712609  2.0526e+07   3000
      0.001 1.60002e+07  0.737109 1.90451e+07   3000
 0.00316228 1.42464e+07  0.765925 1.74253e+07   3000
       0.01 1.29687e+07  0.786917 1.66932e+07   3000
  0.0316228 1.23416e+07  0.797221 1.67489e+07   3000
        0.1 1.20719e+07  0.801653 1.67535e+07   3000
   0.316228 1.20297e+07  0.802346 1.69704e+07   3000
          1         inf      -inf         inf    358


/usr/lib/python3.14/site-packages/numpy/_core/_methods.py:132: RuntimeWarning: overflow encountered in reduce
  ret = umr_sum(arr, axis, dtype, out, keepdims, where=where)
/usr/lib/python3.14/site-packages/numpy/_core/_methods.py:49: RuntimeWarning: overflow encountered in reduce
  return umr_sum(a, axis, dtype, out, keepdims, initial, where)
/usr/lib/python3.14/site-packages/numpy/_core/fromnumeric.py:83: RuntimeWarning: overflow encountered in reduce
  return ufunc.reduce(obj, axis, dtype, out, **passkwargs)
/usr/lib/python3.14/site-packages/numpy/_core/_methods.py:49: RuntimeWarning: overflow encountered in reduce
  return umr_sum(a, axis, dtype, out, keepdims, initial, where)


In [ ]:
best_row_mom        = results_mom_df.loc[results_mom_df['Loss_val'].idxmin()]
best_n_mom          = best_row_mom['n']
best_loss_train_mom = best_row_mom['Loss_train']
best_r2_train_mom   = best_row_mom['R2_train']
best_loss_val_mom   = best_row_mom['Loss_val']

print(f"Лучший n   = {best_n_mom:.6g}")
print(f"Loss_train = {best_loss_train_mom:.6f}")
print(f"R2_train   = {best_r2_train_mom:.4f}")
print(f"Loss_val   = {best_loss_val_mom:.6f}")

Лучший n   = 0.01
Loss_train = 12968736.009682
R2_train   = 0.7869
Loss_val   = 16693193.056509


In [ ]:
best_mom = Momentum(learning_rate=best_n_mom, beta=0.9,
                    n_iters=3000, tol=1e-8)
best_mom.fit(X_train_enc, y_train)

y_test_pred_mom = best_mom.predict(X_test_enc)
if np.all(np.isfinite(y_test_pred_mom)):
    loss_test_mom = float(np.mean((y_test - y_test_pred_mom) ** 2))
    r2_test_mom   = r2_score(y_test, y_test_pred_mom)
else:
    loss_test_mom, r2_test_mom = np.inf, np.nan

n_iter_mom = best_mom.n_iter_done

print("=== Momentum, лучший n ===")
print(f"n          = {best_n_mom:.6g}")
print(f"Loss_train = {best_loss_train_mom:.6f}")
print(f"R2_train   = {best_r2_train_mom:.4f}")
print(f"Loss_val   = {best_loss_val_mom:.6f}")
print(f"Loss_test  = {loss_test_mom:.6f}")
print(f"R2_test    = {r2_test_mom:.4f}")
print(f"Итераций   = {n_iter_mom}")

=== Momentum, лучший n ===
n          = 0.01
Loss_train = 12968736.009682
R2_train   = 0.7869
Loss_val   = 16693193.056509
Loss_test  = 27315761.231371
R2_test    = 0.6083
Итераций   = 3000


In [ ]:
best_mom_model   = best_mom
best_mom_n       = best_n_mom
best_mom_loss_tr = best_loss_train_mom
best_mom_loss_te = loss_test_mom
best_mom_r2_tr   = best_r2_train_mom
best_mom_r2_te   = r2_test_mom
best_mom_iters   = n_iter_mom

11. Исследуйте Momentum с переменным шагом n(lyamda) по формуле TimeDecayLR (из лекции):

переберите lyamda в логарифмической сетке от 10^-5 до 1

для каждого lyamda:

* обучите Momentum на train
* найдите и запомните R^2_train и Loss_train
* протестируйте Momentum на val, запомните Loss_val

найдите наилучший lyamda по минимальному Loss_val, запомните его Loss_train, R^2_train, Loss_val;

протестируйте Momentum c n(best_lyamda) на test, запомните Loss_test, R^2_test, число итераций на test.

In [ ]:
class MomentumTimeDecay:
    def __init__(self, n0=0.1, lam=0.01, beta=0.9, n_iters=3000, tol=1e-8):
        self.n0 = n0
        self.lam = lam
        self.beta = beta
        self.n_iters = n_iters
        self.tol = tol
        self.w = None
        self.loss_history = []
        self.n_iter_done = None

    def fit(self, X, y):
        n_samples, n_features = X.shape
        y = np.asarray(y, dtype=float)
        self.w = np.zeros(n_features)
        v = np.zeros(n_features)
        self.loss_history = []
        self.n_iter_done = self.n_iters

        for t in range(1, self.n_iters + 1):
            lr_t = self.n0 / (1.0 + self.lam * t)   # TimeDecay

            y_pred = np.asarray(X @ self.w).ravel()
            error = y_pred - y
            grad = (2.0 / n_samples) * np.asarray(X.T @ error).ravel()

            v = self.beta * v - lr_t * grad
            self.w += v

            loss = float(np.mean(error ** 2))
            self.loss_history.append(loss)

            if not np.isfinite(loss):
                self.loss_history.append(np.inf)
                self.n_iter_done = t
                break

            if np.linalg.norm(grad) < self.tol:
                self.n_iter_done = t
                break

        return self

    def predict(self, X):
        return np.asarray(X @ self.w).ravel()

    def score(self, X, y):
        y = np.asarray(y, dtype=float)
        y_pred = self.predict(X)
        ss_res = np.sum((y - y_pred) ** 2)
        ss_tot = np.sum((y - y.mean()) ** 2)
        return 1.0 - ss_res / ss_tot

In [ ]:
lambdas = np.logspace(-5, 0, 11)

n0   = 0.1
beta = 0.9


results_mom_td = []

for lam in lambdas:
    model = MomentumTimeDecay(n0=n0, lam=lam, beta=beta,
                              n_iters=3000, tol=1e-8)
    model.fit(X_train_enc, y_train)

    # ---- train ----
    y_train_pred = model.predict(X_train_enc)
    if np.all(np.isfinite(y_train_pred)):
        loss_train = float(np.mean((y_train - y_train_pred) ** 2))
        r2_train   = r2_score(y_train, y_train_pred)
    else:
        loss_train, r2_train = np.inf, np.nan

    # ---- val ----
    y_val_pred = model.predict(X_val_enc)
    if np.all(np.isfinite(y_val_pred)):
        loss_val = float(np.mean((y_val - y_val_pred) ** 2))
    else:
        loss_val = np.inf

    results_mom_td.append({
        'lyamda':     lam,
        'Loss_train': loss_train,
        'R2_train':   r2_train,
        'Loss_val':   loss_val,
        'iters':      len(model.loss_history),
    })

results_mom_td_df = pd.DataFrame(results_mom_td)
pd.set_option('display.float_format', lambda x: f'{x:.6g}')
print(results_mom_td_df.to_string(index=False))

     lyamda  Loss_train  R2_train    Loss_val  iters
      1e-05 1.20735e+07  0.801627 1.67533e+07   3000
3.16228e-05 1.20769e+07  0.801571 1.67531e+07   3000
     0.0001 1.20878e+07  0.801391 1.67535e+07   3000
0.000316228 1.21223e+07  0.800824 1.67583e+07   3000
      0.001 1.22198e+07  0.799222  1.6765e+07   3000
 0.00316228 1.24354e+07   0.79568 1.67275e+07   3000
       0.01 1.28621e+07  0.788669 1.66773e+07   3000
  0.0316228 1.36959e+07  0.774969 1.70136e+07   3000
        0.1 1.49906e+07  0.753696 1.80718e+07   3000
   0.316228  1.6488e+07  0.729094  1.9519e+07   3000
          1  1.7701e+07  0.709163 2.07804e+07   3000


In [ ]:
best_row_mom_td        = results_mom_td_df.loc[results_mom_td_df['Loss_val'].idxmin()]
best_lyamda_mom_td     = best_row_mom_td['lyamda']
best_loss_train_mom_td = best_row_mom_td['Loss_train']
best_r2_train_mom_td   = best_row_mom_td['R2_train']
best_loss_val_mom_td   = best_row_mom_td['Loss_val']

print(f"Лучший lyamda = {best_lyamda_mom_td:.6g}")
print(f"Loss_train    = {best_loss_train_mom_td:.6f}")
print(f"R2_train      = {best_r2_train_mom_td:.4f}")
print(f"Loss_val      = {best_loss_val_mom_td:.6f}")

Лучший lyamda = 0.01
Loss_train    = 12862097.870679
R2_train      = 0.7887
Loss_val      = 16677313.887433


In [ ]:
best_mom_td = MomentumTimeDecay(n0=n0, lam=best_lyamda_mom_td, beta=beta,
                                n_iters=3000, tol=1e-8)
best_mom_td.fit(X_train_enc, y_train)

y_test_pred_mom_td = best_mom_td.predict(X_test_enc)
if np.all(np.isfinite(y_test_pred_mom_td)):
    loss_test_mom_td = float(np.mean((y_test - y_test_pred_mom_td) ** 2))
    r2_test_mom_td   = r2_score(y_test, y_test_pred_mom_td)
else:
    loss_test_mom_td, r2_test_mom_td = np.inf, np.nan

n_iter_mom_td = best_mom_td.n_iter_done

print("=== Momentum + TimeDecayLR ===")
print(f"n0            = {n0}")
print(f"beta          = {beta}")
print(f"best lyamda   = {best_lyamda_mom_td:.6g}")
print(f"Loss_train    = {best_loss_train_mom_td:.6f}")
print(f"R2_train      = {best_r2_train_mom_td:.4f}")
print(f"Loss_val      = {best_loss_val_mom_td:.6f}")
print(f"Loss_test     = {loss_test_mom_td:.6f}")
print(f"R2_test       = {r2_test_mom_td:.4f}")
print(f"Итераций      = {n_iter_mom_td}")

=== Momentum + TimeDecayLR ===
n0            = 0.1
beta          = 0.9
best lyamda   = 0.01
Loss_train    = 12862097.870679
R2_train      = 0.7887
Loss_val      = 16677313.887433
Loss_test     = 27462415.118280
R2_test       = 0.6062
Итераций      = 3000


In [ ]:
best_mom_td_model   = best_mom_td
best_mom_td_lyamda  = best_lyamda_mom_td
best_mom_td_loss_tr = best_loss_train_mom_td
best_mom_td_loss_te = loss_test_mom_td
best_mom_td_r2_tr   = best_r2_train_mom_td
best_mom_td_r2_te   = r2_test_mom_td
best_mom_td_iters   = n_iter_mom_td

12. Исследуйте Adam с постоянным шагом n:

переберите n в логарифмической сетке от 10^-5 до 1

для каждого n:

* обучите Adam на train
* найдите и запомните R^2_train и Loss_train
* протестируйте Adam на val, запомните Loss_val

найдите наилучший n по минимальному Loss_val, запомните его Loss_train, R^2_train, Loss_val;

протестируйте Adam c лучшим n на test, запомните Loss_test, R^2_test, число итераций на test.

In [ ]:
class Adam:
    def __init__(self, learning_rate=0.01, beta1=0.9, beta2=0.999,
                 eps=1e-8, n_iters=3000, tol=1e-8):
        self.learning_rate = learning_rate
        self.beta1 = beta1
        self.beta2 = beta2
        self.eps = eps
        self.n_iters = n_iters
        self.tol = tol
        self.w = None
        self.loss_history = []
        self.n_iter_done = None

    def fit(self, X, y):
        n_samples, n_features = X.shape
        y = np.asarray(y, dtype=float)
        self.w = np.zeros(n_features)
        m = np.zeros(n_features)
        v = np.zeros(n_features)
        self.loss_history = []
        self.n_iter_done = self.n_iters

        for t in range(1, self.n_iters + 1):
            y_pred = np.asarray(X @ self.w).ravel()
            error = y_pred - y
            grad = (2.0 / n_samples) * np.asarray(X.T @ error).ravel()

            m = self.beta1 * m + (1.0 - self.beta1) * grad
            v = self.beta2 * v + (1.0 - self.beta2) * (grad ** 2)

            m_hat = m / (1.0 - self.beta1 ** t)
            v_hat = v / (1.0 - self.beta2 ** t)

            self.w -= self.learning_rate * m_hat / (np.sqrt(v_hat) + self.eps)

            loss = float(np.mean(error ** 2))
            self.loss_history.append(loss)

            if not np.isfinite(loss):
                self.loss_history.append(np.inf)
                self.n_iter_done = t
                break

            if np.linalg.norm(grad) < self.tol:
                self.n_iter_done = t
                break

        return self

    def predict(self, X):
        return np.asarray(X @ self.w).ravel()

    def score(self, X, y):
        y = np.asarray(y, dtype=float)
        y_pred = self.predict(X)
        ss_res = np.sum((y - y_pred) ** 2)
        ss_tot = np.sum((y - y.mean()) ** 2)
        return 1.0 - ss_res / ss_tot

In [117]:
lambdas = np.logspace(-5, 0, 11)

results_adam = []

for n in lambdas:
    model = Adam(learning_rate=n, beta1=0.9, beta2=0.999,
                 eps=1e-8, n_iters=3000, tol=1e-8)
    model.fit(X_train_enc, y_train)

    # ---- train ----
    y_train_pred = model.predict(X_train_enc)
    if np.all(np.isfinite(y_train_pred)):
        loss_train = float(np.mean((y_train - y_train_pred) ** 2))
        r2_train   = r2_score(y_train, y_train_pred)
    else:
        loss_train, r2_train = np.inf, np.nan

    # ---- val ----
    y_val_pred = model.predict(X_val_enc)
    if np.all(np.isfinite(y_val_pred)):
        loss_val = float(np.mean((y_val - y_val_pred) ** 2))
    else:
        loss_val = np.inf

    results_adam.append({
        'n':          n,
        'Loss_train': loss_train,
        'R2_train':   r2_train,
        'Loss_val':   loss_val,
        'iters':      len(model.loss_history),
    })

results_adam_df = pd.DataFrame(results_adam)
pd.set_option('display.float_format', lambda x: f'{x:.6g}')
print(results_adam_df.to_string(index=False))

          n  Loss_train  R2_train    Loss_val  iters
      1e-05 1.09273e+08 -0.795404 1.07209e+08   3000
3.16228e-05 1.09266e+08 -0.795289 1.07202e+08   3000
     0.0001 1.09243e+08 -0.794925  1.0718e+08   3000
0.000316228 1.09173e+08 -0.793774 1.07109e+08   3000
      0.001 1.08952e+08 -0.790141 1.06886e+08   3000
 0.00316228 1.08256e+08 -0.778701 1.06184e+08   3000
       0.01 1.06084e+08  -0.74301 1.03992e+08   3000
  0.0316228 9.95063e+07 -0.634938 9.73525e+07   3000
        0.1 8.14682e+07 -0.338563 7.91067e+07   3000
   0.316228 4.61747e+07  0.241327 4.34149e+07   3000
          1 1.86496e+07  0.693577 1.90846e+07   3000


In [118]:
best_row_adam        = results_adam_df.loc[results_adam_df['Loss_val'].idxmin()]
best_n_adam          = best_row_adam['n']
best_loss_train_adam = best_row_adam['Loss_train']
best_r2_train_adam   = best_row_adam['R2_train']
best_loss_val_adam   = best_row_adam['Loss_val']

print(f"Лучший n   = {best_n_adam:.6g}")
print(f"Loss_train = {best_loss_train_adam:.6f}")
print(f"R2_train   = {best_r2_train_adam:.4f}")
print(f"Loss_val   = {best_loss_val_adam:.6f}")

Лучший n   = 1
Loss_train = 18649636.584476
R2_train   = 0.6936
Loss_val   = 19084610.399194


In [119]:
best_adam = Adam(learning_rate=best_n_adam, beta1=0.9, beta2=0.999,
                 eps=1e-8, n_iters=3000, tol=1e-8)
best_adam.fit(X_train_enc, y_train)

y_test_pred_adam = best_adam.predict(X_test_enc)
if np.all(np.isfinite(y_test_pred_adam)):
    loss_test_adam = float(np.mean((y_test - y_test_pred_adam) ** 2))
    r2_test_adam   = r2_score(y_test, y_test_pred_adam)
else:
    loss_test_adam, r2_test_adam = np.inf, np.nan

n_iter_adam = best_adam.n_iter_done

print("=== Adam, лучший n ===")
print(f"n          = {best_n_adam:.6g}")
print(f"Loss_train = {best_loss_train_adam:.6f}")
print(f"R2_train   = {best_r2_train_adam:.4f}")
print(f"Loss_val   = {best_loss_val_adam:.6f}")
print(f"Loss_test  = {loss_test_adam:.6f}")
print(f"R2_test    = {r2_test_adam:.4f}")
print(f"Итераций   = {n_iter_adam}")

=== Adam, лучший n ===
n          = 1
Loss_train = 18649636.584476
R2_train   = 0.6936
Loss_val   = 19084610.399194
Loss_test  = 28975255.864518
R2_test    = 0.5845
Итераций   = 3000


In [121]:
best_adam_model   = best_adam
best_adam_n       = best_n_adam
best_adam_loss_tr = best_loss_train_adam
best_adam_loss_te = loss_test_adam
best_adam_r2_tr   = best_r2_train_adam
best_adam_r2_te   = r2_test_adam
best_adam_iters   = n_iter_adam

13. Исследуйте Adam с переменным шагом n(lyamda) по формуле TimeDecayLR (из лекции):

переберите lyamda в логарифмической сетке от 10^-5 до 1

для каждого lyamda:

* обучите Adam на train
* найдите и запомните R^2_train и Loss_train
* протестируйте Adam на val, запомните Loss_val

найдите наилучший lyamda по минимальному Loss_val, запомните его Loss_train, R^2_train, Loss_val;

протестируйте Adam c n(best_lyamda) на test, запомните Loss_test, R^2_test, число итераций на test.

In [ ]:
class AdamTimeDecay:
    def __init__(self, n0=0.1, lam=0.01, beta1=0.9, beta2=0.999,
                 eps=1e-8, n_iters=3000, tol=1e-8):
        self.n0 = n0
        self.lam = lam
        self.beta1 = beta1
        self.beta2 = beta2
        self.eps = eps
        self.n_iters = n_iters
        self.tol = tol
        self.w = None
        self.loss_history = []
        self.n_iter_done = None

    def fit(self, X, y):
        n_samples, n_features = X.shape
        y = np.asarray(y, dtype=float)
        self.w = np.zeros(n_features)
        m = np.zeros(n_features)
        v = np.zeros(n_features)
        self.loss_history = []
        self.n_iter_done = self.n_iters

        for t in range(1, self.n_iters + 1):
            lr_t = self.n0 / (1.0 + self.lam * t)   # TimeDecay

            y_pred = np.asarray(X @ self.w).ravel()
            error = y_pred - y
            grad = (2.0 / n_samples) * np.asarray(X.T @ error).ravel()

            m = self.beta1 * m + (1.0 - self.beta1) * grad
            v = self.beta2 * v + (1.0 - self.beta2) * (grad ** 2)

            m_hat = m / (1.0 - self.beta1 ** t)
            v_hat = v / (1.0 - self.beta2 ** t)

            self.w -= lr_t * m_hat / (np.sqrt(v_hat) + self.eps)

            loss = float(np.mean(error ** 2))
            self.loss_history.append(loss)

            if not np.isfinite(loss):
                self.loss_history.append(np.inf)
                self.n_iter_done = t
                break

            if np.linalg.norm(grad) < self.tol:
                self.n_iter_done = t
                break

        return self

    def predict(self, X):
        return np.asarray(X @ self.w).ravel()

    def score(self, X, y):
        y = np.asarray(y, dtype=float)
        y_pred = self.predict(X)
        ss_res = np.sum((y - y_pred) ** 2)
        ss_tot = np.sum((y - y.mean()) ** 2)
        return 1.0 - ss_res / ss_tot

In [113]:
lambdas = np.logspace(-5, 0, 11)

n0   = 0.1
beta1 = 0.9
beta2 = 0.999

results_adam_td = []

for lam in lambdas:
    model = AdamTimeDecay(n0=n0, lam=lam, beta1=beta1, beta2=beta2,
                          eps=1e-8, n_iters=3000, tol=1e-8)
    model.fit(X_train_enc, y_train)

    # ---- train ----
    y_train_pred = model.predict(X_train_enc)
    if np.all(np.isfinite(y_train_pred)):
        loss_train = float(np.mean((y_train - y_train_pred) ** 2))
        r2_train   = r2_score(y_train, y_train_pred)
    else:
        loss_train, r2_train = np.inf, np.nan

    # ---- val ----
    y_val_pred = model.predict(X_val_enc)
    if np.all(np.isfinite(y_val_pred)):
        loss_val = float(np.mean((y_val - y_val_pred) ** 2))
    else:
        loss_val = np.inf

    results_adam_td.append({
        'lyamda':     lam,
        'Loss_train': loss_train,
        'R2_train':   r2_train,
        'Loss_val':   loss_val,
        'iters':      len(model.loss_history),
    })

results_adam_td_df = pd.DataFrame(results_adam_td)
pd.set_option('display.float_format', lambda x: f'{x:.6g}')
print(results_adam_td_df.to_string(index=False))

     lyamda  Loss_train  R2_train    Loss_val  iters
      1e-05  8.1812e+07 -0.344212 7.94549e+07   3000
3.16228e-05 8.25173e+07   -0.3558 8.01693e+07   3000
     0.0001 8.44608e+07 -0.387734 8.21375e+07   3000
0.000316228 8.87712e+07 -0.458555 8.65002e+07   3000
      0.001 9.52885e+07 -0.565637 9.30911e+07   3000
 0.00316228 1.01533e+08  -0.66824 9.93995e+07   3000
       0.01 1.05631e+08 -0.735564 1.03535e+08   3000
  0.0316228 1.07733e+08 -0.770103 1.05656e+08   3000
        0.1 1.08666e+08 -0.785441 1.06598e+08   3000
   0.316228 1.09047e+08 -0.791697 1.06981e+08   3000
          1 1.09194e+08 -0.794112  1.0713e+08   3000


In [114]:
best_row_adam_td        = results_adam_td_df.loc[results_adam_td_df['Loss_val'].idxmin()]
best_lyamda_adam_td     = best_row_adam_td['lyamda']
best_loss_train_adam_td = best_row_adam_td['Loss_train']
best_r2_train_adam_td   = best_row_adam_td['R2_train']
best_loss_val_adam_td   = best_row_adam_td['Loss_val']

print(f"Лучший lyamda = {best_lyamda_adam_td:.6g}")
print(f"Loss_train    = {best_loss_train_adam_td:.6f}")
print(f"R2_train      = {best_r2_train_adam_td:.4f}")
print(f"Loss_val      = {best_loss_val_adam_td:.6f}")

Лучший lyamda = 1e-05
Loss_train    = 81811968.612980
R2_train      = -0.3442
Loss_val      = 79454882.852475


In [115]:
best_adam_td = AdamTimeDecay(n0=n0, lam=best_lyamda_adam_td,
                             beta1=beta1, beta2=beta2,
                             eps=1e-8, n_iters=3000, tol=1e-8)
best_adam_td.fit(X_train_enc, y_train)

y_test_pred_adam_td = best_adam_td.predict(X_test_enc)
if np.all(np.isfinite(y_test_pred_adam_td)):
    loss_test_adam_td = float(np.mean((y_test - y_test_pred_adam_td) ** 2))
    r2_test_adam_td   = r2_score(y_test, y_test_pred_adam_td)
else:
    loss_test_adam_td, r2_test_adam_td = np.inf, np.nan

n_iter_adam_td = best_adam_td.n_iter_done

print("=== Adam + TimeDecayLR ===")
print(f"n0            = {n0}")
print(f"beta1, beta2  = {beta1}, {beta2}")
print(f"best lyamda   = {best_lyamda_adam_td:.6g}")
print(f"Loss_train    = {best_loss_train_adam_td:.6f}")
print(f"R2_train      = {best_r2_train_adam_td:.4f}")
print(f"Loss_val      = {best_loss_val_adam_td:.6f}")
print(f"Loss_test     = {loss_test_adam_td:.6f}")
print(f"R2_test       = {r2_test_adam_td:.4f}")
print(f"Итераций      = {n_iter_adam_td}")

=== Adam + TimeDecayLR ===
n0            = 0.1
beta1, beta2  = 0.9, 0.999
best lyamda   = 1e-05
Loss_train    = 81811968.612980
R2_train      = -0.3442
Loss_val      = 79454882.852475
Loss_test     = 99600298.521072
R2_test       = -0.4282
Итераций      = 3000


In [116]:
best_adam_td_model   = best_adam_td
best_adam_td_lyamda  = best_lyamda_adam_td
best_adam_td_loss_tr = best_loss_train_adam_td
best_adam_td_loss_te = loss_test_adam_td
best_adam_td_r2_tr   = best_r2_train_adam_td
best_adam_td_r2_te   = r2_test_adam_td
best_adam_td_iters   = n_iter_adam_td

14. Постройте итоговую сравнительную таблицу со следующими столбцами:

1) название метода

2) значение лучшего шага (если n) или функция лучшего шага (если n(lyamda))

3) Loss_train

4) Loss_test

5) R^2 train

6) R^2 test

7) число итераций на test

In [123]:
import pandas as pd
import numpy as np

# Переводим MSE в RMSE (корень)
rmse_train = lambda mse: np.sqrt(mse) if np.isfinite(mse) else np.inf
rmse_test  = lambda mse: np.sqrt(mse) if np.isfinite(mse) else np.inf

comparison = pd.DataFrame({
    'Метод': [
        'VGD (const n)',
        'VGD + TimeDecay',
        'SGD (const n)',
        'SGD + TimeDecay',
        'SAG (const n)',
        'SAG + TimeDecay',
        'Momentum (const n)',
        'Momentum + TimeDecay',
        'Adam (const n)',
        'Adam + TimeDecay',
    ],
    'Лучший шаг': [
        f"n = {best_n_vgd:.4g}",
        f"n(t) = {n0:.3g} / (1 + {best_lyamda_td:.3g}·t)",
        f"n = {best_n_sgd:.4g}",
        f"n(t) = {n0:.3g} / (1 + {best_lyamda_sgd_td:.3g}·t)",
        f"n = {best_n_sag:.4g}",
        f"n(t) = {n0:.3g} / (1 + {best_lyamda_sag_td:.3g}·t)",
        f"n = {best_n_mom:.4g}",
        f"n(t) = {n0:.3g} / (1 + {best_lyamda_mom_td:.3g}·t)",
        f"n = {best_n_adam:.4g}",
        f"n(t) = {n0:.3g} / (1 + {best_lyamda_adam_td:.3g}·t)",
    ],
    'RMSE_train': [
        rmse_train(best_vgd_loss_tr),
        rmse_train(best_td_loss_tr),
        rmse_train(best_sgd_loss_tr),
        rmse_train(best_sgd_td_loss_tr),
        rmse_train(best_sag_loss_tr),
        rmse_train(best_sag_td_loss_tr),
        rmse_train(best_mom_loss_tr),
        rmse_train(best_mom_td_loss_tr),
        rmse_train(best_adam_loss_tr),
        rmse_train(best_adam_td_loss_tr),
    ],
    'RMSE_test': [
        rmse_test(best_vgd_loss_te),
        rmse_test(best_td_loss_te),
        rmse_test(best_sgd_loss_te),
        rmse_test(best_sgd_td_loss_te),
        rmse_test(best_sag_loss_te),
        rmse_test(best_sag_td_loss_te),
        rmse_test(best_mom_loss_te),
        rmse_test(best_mom_td_loss_te),
        rmse_test(best_adam_loss_te),
        rmse_test(best_adam_td_loss_te),
    ],
    'R²_train': [
        best_vgd_r2_tr,  best_td_r2_tr,
        best_sgd_r2_tr,  best_sgd_td_r2_tr,
        best_sag_r2_tr,  best_sag_td_r2_tr,
        best_mom_r2_tr,  best_mom_td_r2_tr,
        best_adam_r2_tr, best_adam_td_r2_tr,
    ],
    'R²_test': [
        best_vgd_r2_te,  best_td_r2_te,
        best_sgd_r2_te,  best_sgd_td_r2_te,
        best_sag_r2_te,  best_sag_td_r2_te,
        best_mom_r2_te,  best_mom_td_r2_te,
        best_adam_r2_te, best_adam_td_r2_te,
    ],
    'Итераций на test': [
        best_vgd_iters,  best_td_iters,
        best_sgd_iters,  best_sgd_td_iters,
        best_sag_iters,  best_sag_td_iters,
        best_mom_iters,  best_mom_td_iters,
        best_adam_iters, best_adam_td_iters,
    ],
})

pd.set_option('display.float_format', lambda x: f'{x:,.2f}')
pd.set_option('display.width', 220)
pd.set_option('display.max_colwidth', 45)

print("Таблица 1. Сравнение методов градиентного спуска")
print(comparison.to_string(index=False))

Таблица 1. Сравнение методов градиентного спуска
               Метод                    Лучший шаг  RMSE_train  RMSE_test  R²_train  R²_test  Итераций на test
       VGD (const n)                       n = 0.1    3,601.16   5,226.20      0.79     0.61              3000
     VGD + TimeDecay    n(t) = 0.1 / (1 + 1e-05·t)    3,602.82   5,224.73      0.79     0.61              3000
       SGD (const n)                      n = 0.01    3,706.44   5,477.84      0.77     0.57                50
     SGD + TimeDecay n(t) = 0.1 / (1 + 0.000316·t)    3,639.96   5,307.12      0.78     0.60                50
       SAG (const n)                     n = 0.001    3,944.36   5,013.65      0.74     0.64                50
     SAG + TimeDecay  n(t) = 0.1 / (1 + 0.00316·t)    3,775.62   5,189.94      0.77     0.61                50
  Momentum (const n)                      n = 0.01    3,601.21   5,226.45      0.79     0.61              3000
Momentum + TimeDecay     n(t) = 0.1 / (1 + 0.01·t)    3,586.38 

15. Сделайте вывод о том, какой метод и шаг линейной регрессии самый лучший для данной выборки и ответьте на вопросы:

1) почему именно этот метод и этот шаг самый лучший (по каким данным из таблицы вы сделали такой вывод)

2) расскажите простыми словами суть R^2_train и R^2_test?

3) как R^2_train и R^2_test помогают сравнивать методы? почему оба эти значения надо вычислять для данного выбора?

# ОТВЕТ

Из таблички видно, что лучший метод SAG c постоянным шагом 0.001. 
1) Такой вывод можно сделать исходя их того, что у него минимальный loss и максимальный R2 test
2) R2 train показывает на сколько модель хорошо запомнила обучающую выборку, R2 test показывает на сколько хорошо моделт предсказывает
3) Чем больше R2 у определенного метода - тем он лучше по сравнению с другими. Они нужны для понимания того, переобучилась ли модель(если разрыв большой, то переобучение).
4) R2 отрицательный, потому что модель плохо обучилась